In [43]:
!pip3 install investpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [44]:
import os
from dotenv import load_dotenv
from fredapi import Fred
import yfinance as yf
import pandas as pd
import numpy as np


Before running the cell below, you should create an account on [FRED's Website](https://fred.stlouisfed.org/docs/api/api_key.html) and create an API key. Store the key in a local .env file like this `FRED_API_KEY=your_actual_key_here_12345`. Note you do not need to run this file as all output data as been stored in raw_data.csv

In [45]:
# 1. SETUP
load_dotenv()
fred = Fred(api_key=os.getenv('FRED_API_KEY', 'edf185d621148e61f50e97f856494963'))


In [46]:
tickers = ['^GSPC', '^VIX', 'SPY']
market_raw = yf.download(tickers, start='1990-01-02', progress=False)
market_data = pd.DataFrame({
    'GSPC': market_raw['Close']['^GSPC'],
    'VIX': market_raw['Close']['^VIX'],
    'SPY Volume': market_raw['Volume']['SPY']
})

# 2.5. FETCH COMMODITY DATA (INVESTING.COM -> YAHOO FALLBACK)
oil_data = fred.get_series('DCOILWTICO')
commodity_data = pd.DataFrame({'Oil': oil_data})
commodity_data = commodity_data.resample('D').ffill()
print('Commodity source: FRED (DCOILWTICO)')

commodity_data.index = pd.to_datetime(commodity_data.index)

# 3. FETCH MACRO DATA (FRED)
macro_data = pd.DataFrame({
    'GDP': fred.get_series('GDPC1'),
    'Core_Inflation': fred.get_series('PCEPILFE'),
    'Unemployment': fred.get_series('UNRATE'),
    'M2': fred.get_series('M2SL'),
    'Sentiment': fred.get_series('UMCSENT')
})

# Align data to daily frequency and forward-fill gaps
macro_data = macro_data.resample('D').ffill()



# 4. JOIN AND SAVE TO CSV
raw_data = market_data.join(macro_data, how='left')
raw_data = market_data.join(commodity_data, how='left').join(macro_data, how='left')
raw_data = raw_data.ffill().dropna()
raw_data.to_csv('../data/raw_data.csv')
print(f"Ingestion Complete. Shape: {raw_data.shape}. File: ../data/raw_data.csv")

Commodity source: FRED (DCOILWTICO)
Ingestion Complete. Shape: (8359, 9). File: ../data/raw_data.csv


In [47]:
# Check earliest date for each series
print("=== Data Start Dates ===")
print(f"GSPC:           {market_data['GSPC'].first_valid_index()}")
print(f"VIX:            {market_data['VIX'].first_valid_index()}")
print(f"SPY Volume:     {market_data['SPY Volume'].first_valid_index()}")
print(f"Oil:            {commodity_data['Oil'].first_valid_index()}")
print(f"GDP:            {fred.get_series('GDPC1').first_valid_index()}")
print(f"Core_Inflation: {fred.get_series('PCEPILFE').first_valid_index()}")
print(f"Unemployment:   {fred.get_series('UNRATE').first_valid_index()}")
print(f"M2:             {fred.get_series('M2SL').first_valid_index()}")
print(f"Sentiment:      {fred.get_series('UMCSENT').first_valid_index()}")
print("========================")

=== Data Start Dates ===
GSPC:           1990-01-02 00:00:00
VIX:            1990-01-02 00:00:00
SPY Volume:     1993-01-29 00:00:00
Oil:            1986-01-02 00:00:00
GDP:            1947-01-01 00:00:00
Core_Inflation: 1959-01-01 00:00:00
Unemployment:   1948-01-01 00:00:00
M2:             1959-01-01 00:00:00
Sentiment:      1952-11-01 00:00:00


In [39]:
market_data

,GSPC,VIX,SPY Volume
Date,,,
1999-01-04,1228.099976,26.170000,9450400
1999-01-05,1244.780029,24.459999,8031000
1999-01-06,1272.339966,23.340000,7737700
1999-01-07,1269.729980,24.370001,5504900
1999-01-08,1275.089966,23.280001,6224400
...,...,...,...
2026-04-09,6824.660156,19.490000,57134400
2026-04-10,6816.890137,19.230000,42253500
2026-04-13,6886.240234,19.120001,54185800
